# Experiment 03 — Longitudinal Routing (24h, 5 ISPs × 5 sites)

Loads the final snapshot of `run_20260611_24h` and reproduces the charts behind
`findings/03_longitudinal_routing_24h.md`: per-ISP RTT, diurnal stability, local vs
offshore, the overnight **thunderstorm outage**, ping loss/jitter, and path stability.

Run top-to-bottom. Point `RESULTS` at a different run folder to re-use on future runs.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = '../experiments/03_longitudinal_routing/results/run_20260611_24h'
ts_file = sorted(glob.glob(os.path.join(RESULTS, 'trace_summary_2026*.csv')))[-1]
pg_file = sorted(glob.glob(os.path.join(RESULTS, 'ping_series_2026*.csv')))[-1]
print('trace:', os.path.basename(ts_file))
print('ping :', os.path.basename(pg_file))

df = pd.read_csv(ts_file)
df['t'] = pd.to_datetime(df['trace_time'])
df['hour'] = df['t'].dt.hour

pg = pd.read_csv(pg_file)
pg['loss'] = 100 * (pg['sent'] - pg['rcvd']) / pg['sent']

PROBE_ORDER = ['Nayatel', 'Transworld', 'Z COM Networks', 'Cybernet', 'LocalInternetProj01']
ok = df[df['destination_responded']]            # rounds that reached the destination
print('trace rounds:', len(df), '| pings:', len(pg),
      '| span:', df['t'].min(), '->', df['t'].max())

## 1. Median RTT to each site, by ISP

PTCL is absent here — it firewalls ICMP at the host (0% replies), so it has no RTT.
Note the huge ISP gap on the *local* sites vs the ISP-agnostic offshore banks.

In [ ]:
piv = ok.pivot_table(index='target_label', columns='probe_city',
                     values='dest_rtt_ms', aggfunc='median').round(1)
piv = piv.reindex(columns=PROBE_ORDER)
display(piv)

ax = piv.plot(kind='bar', figsize=(11, 5))
ax.set_ylabel('median RTT (ms)')
ax.set_title('Median RTT to each site, by ISP (24h)')
ax.legend(title='ISP', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 2. Diurnal RTT (fixed probe)

Hold one *clean* probe fixed (so the flaky LocalInternetProj01 doesn't skew the
aggregate) and plot median RTT per hour. The lines are essentially flat — **no
peak-evening congestion cycle**; the offshore penalty is structural, not time-of-day.

In [ ]:
def diurnal(probe='Nayatel'):
    sub = ok[ok['probe_city'] == probe]
    fig, ax = plt.subplots(figsize=(11, 5))
    for s in sorted(sub['target_label'].unique()):
        h = sub[sub['target_label'] == s].groupby('hour')['dest_rtt_ms'].median()
        ax.plot(h.index, h.values, marker='o', ms=3, label=s)
    ax.set_xlabel('hour of day (UTC)'); ax.set_ylabel('median RTT (ms)')
    ax.set_title(f'Diurnal RTT by hour — probe: {probe}')
    ax.set_xticks(range(0, 24, 2)); ax.legend()
    plt.tight_layout(); plt.show()

diurnal('Nayatel')   # try 'Z COM Networks', 'Transworld', 'Cybernet'

## 3. Local vs offshore

The whole-experiment contrast: PK-hosted sites are 2–42 ms (ISP-dependent), the two
banks 127–211 ms because they're hosted abroad (MCB→Singapore, HBL→New Jersey).

In [ ]:
host = {'dunyanews.tv': 'PK local', 'pseb.org.pk': 'PK local',
        'mcb.com.pk': 'offshore (SG)', 'hbl.com': 'offshore (US)'}
ok2 = ok.assign(host=ok['target_hostname'].map(host)).dropna(subset=['host'])
print(ok2.groupby('host')['dest_rtt_ms'].median().round(1))

ok2.boxplot(column='dest_rtt_ms', by='host', figsize=(8, 5))
plt.suptitle(''); plt.title('Local vs offshore dest RTT (all ISPs pooled)')
plt.ylabel('dest RTT (ms)'); plt.xlabel('')
plt.tight_layout(); plt.show()

## 4. Outages — overnight thunderstorm

When an ISP loses internet, its probe produces **no result to any site**, so an outage
is a stretch of missing rounds across all targets at once. `rounds/hour` per probe
(≈20 when up, 0 when down) and a gap scan pinpoint it: **LocalInternetProj01
(TPCPL/Nova) was down ~5.7 h (01:30–07:15 UTC = 06:30–12:15 PKT, June 12)**; the other
four stayed continuous.

In [ ]:
df['hr'] = df['t'].dt.floor('h')
ct = (df.groupby(['hr', 'probe_city']).size()
        .unstack('probe_city').reindex(columns=PROBE_ORDER).fillna(0).astype(int))
display(ct)   # ~20 = probe up, 0 = ISP outage

ax = ct.plot(figsize=(12, 5))
ax.axhline(20, ls='--', c='grey', lw=0.8)
ax.set_ylabel('rounds per hour'); ax.set_xlabel('hour (UTC)')
ax.set_title('Probe liveness — dips to 0 = ISP outage (thunderstorm)')
ax.legend(title='ISP', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()

print('Outage windows (gap > 20 min between consecutive rounds):')
for probe, g in df.groupby('probe_city'):
    tps = g['t'].sort_values().drop_duplicates().reset_index(drop=True)
    for i in range(1, len(tps)):
        d = (tps[i] - tps[i-1]).total_seconds() / 60
        if d > 20:
            print(f'  {probe:<20} DOWN {tps[i-1]:%m-%d %H:%M} -> UP {tps[i]:%m-%d %H:%M} UTC ({d/60:.1f} h)')

## 5. Ping loss & jitter (data quality)

Loss here includes PTCL (100%, ICMP-blocked) and LocalInternetProj01's storm outage,
so read it as a data-quality view, not pure network loss. The four non-LocalIP01
probes are clean (<1–3%); LocalIP01 is the flaky one.

In [ ]:
# exclude PTCL (always 100% ICMP loss) for a fairer probe-quality view
pg_ok = pg[pg['target_label'] != 'PTCL']
pstat = (pg_ok.groupby('probe_city')
               .agg(loss_pct=('loss', 'mean'), jitter_ms=('rtt_avg', 'std'))
               .reindex(PROBE_ORDER).round(2))
display(pstat)

pstat['loss_pct'].plot(kind='bar', figsize=(8, 4), color='indianred')
plt.ylabel('mean ping loss % (excl. PTCL)')
plt.title('Ping loss by probe — LocalInternetProj01 = storm outage + path flakiness')
plt.tight_layout(); plt.show()

## 6. Path stability

Over 24 h the AS-paths barely moved. The few flagged pairs are **last-hop visibility
flicker** (Sucuri / PTCL final hop intermittently replying), not BGP route changes —
and because we use Paris traceroute, a stable string means a genuinely stable route.

In [ ]:
rows = []
for (site, probe), g in df.sort_values('t').groupby(['target_label', 'probe_city']):
    p = g['asns_in_path'].dropna()
    ch = int((p.values[1:] != p.values[:-1]).sum()) if len(p) > 1 else 0
    rows.append({'site': site, 'probe': probe,
                 'distinct_paths': int(p.nunique()), 'changes': ch})
pc = pd.DataFrame(rows)
print('pairs with a flagged change:', int((pc['changes'] > 0).sum()), 'of', len(pc))
display(pc[pc['changes'] > 0].reset_index(drop=True))